In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [ ]:
import json
from uuid import uuid4
from pathlib import Path
from textwrap import dedent

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema

## Data Loading

In [3]:
df = pd.read_json("../data/corpus.jsonl", lines=True)
df.head()

,id,title,content,published_at,word_count,source_url
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...,2023-07-24,66,https://basasunda.com/puisi-bahasa-sunda
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...,2023-07-24,94,https://basasunda.com/puisi-bahasa-sunda
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...,2023-07-24,50,https://basasunda.com/puisi-bahasa-sunda
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...,2023-07-24,53,https://basasunda.com/puisi-bahasa-sunda
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...,2023-07-24,77,https://basasunda.com/puisi-bahasa-sunda


## Synthetic Data Generation using LLMs

In [4]:
client = OpenAI()

In [31]:
SYSTEM_PROMPTS = {
    "BEIR": dedent("""
                    Generate 5 short search query-answer pairs based on the provided document. Each query should resemble a natural search input (as if searching on Google or other search engines).
                    - The query and the answer must be written in Sundanese.
                    - Ensure the response is concise, accurate, and directly relevant to the document.
                    """).strip(),
    "TRIPLET": dedent("""
                    Create an MS-MARCO triplet dataset from the given document. Each triplet consists of:

                    1. Query: A short, natural search query based on the document.
                    2. Positive Passage: A passage from the document that directly answers the query.
                    3. Negative Passage: A passage from the document that does not answer the query but is still related to the topic.

                    Requirements:

                    - Generate exactly 5 triplets.
                    - Write all elements (query, positive passage, negative passage) in Sundanese.
                    - Ensure the query and passages are concise and coherent.
                    - Each passage must be self-contained and self-explanatory with context included
                    """).strip(),
}

### Synthetic BEIR

In [6]:
class BEIRQueryItem(BaseModel):
    search_term: str
    answer: str


class BEIRQuery(BaseModel):
    queries: list[BEIRQueryItem]

In [9]:
completion_beir = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRQuery,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["BEIR"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 2],
        },
    ],
)

In [11]:
print(completion_beir)

ParsedChatCompletion[BEIRQuery](
    id='chatcmpl-BFxv342rSEccHKHCIJ0EKQu29MXvf',
    choices=[
        ParsedChoice[BEIRQuery](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRQuery](
                content='{"queries":[{"search_term":"Naon hartina bismillah dina lampah?","answer":"Bismillah 
hartina ngamimitian hiji hal kalayan nginget ka Gusti."},{"search_term":"Kumaha cara ngudag kahayang anu 
bener?","answer":"Ngudag kahayang anu bener téh kudu sabar, usaha, jeung ampir hiji-hiji."},{"search_term":"Kumaha 
carana ulah sombong?","answer":"Ulah sombong ku cara hormat ka batur jeung nginget asal-usul 
diri."},{"search_term":"Naon anu kedah dilakukeun lamun gering?","answer":"Lamun gering, kudu jaga kaséhatan jeung 
laun-laun mulih ka cageur."},{"search_term":"Kumaha cara nangkep nu sajati?","answer":"Cara nangkep nu sajati 
nyaéta ku cara pariksa ati sareng mikir saluyu."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRQuery(
                    queries=[
                        BEIRQueryItem(
                            search_term='Naon hartina bismillah dina lampah?',
                            answer='Bismillah hartina ngamimitian hiji hal kalayan nginget ka Gusti.'
                        ),
                        BEIRQueryItem(
                            search_term='Kumaha cara ngudag kahayang anu bener?',
                            answer='Ngudag kahayang anu bener téh kudu sabar, usaha, jeung ampir hiji-hiji.'
                        ),
                        BEIRQueryItem(
                            search_term='Kumaha carana ulah sombong?',
                            answer='Ulah sombong ku cara hormat ka batur jeung nginget asal-usul diri.'
                        ),
                        BEIRQueryItem(
                            search_term='Naon anu kedah dilakukeun lamun gering?',
                            answer='Lamun gering, kudu jaga kaséhatan jeung laun-laun mulih ka cageur.'
                        ),
                        BEIRQueryItem(
                            search_term='Kumaha cara nangkep nu sajati?',
                            answer='Cara nangkep nu sajati nyaéta ku cara pariksa ati sareng mikir saluyu.'
                        )
                    ]
                ),
                annotations=[]
            )
        )
    ],
    created=1743144949,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_b376dfbbd5',
    usage=CompletionUsage(
        completion_tokens=192,
        prompt_tokens=282,
        total_tokens=474,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

### Synthetic Triplet

In [12]:
class TripletItem(BaseModel):
    query: str
    positive_passage: str
    negative_passage: str


class TripetData(BaseModel):
    triplets: list[TripletItem]

In [ ]:
completion_marco = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=TripetData,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["TRIPLET"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 2],
        },
    ],
)

In [17]:
print(completion_marco)

ParsedChatCompletion[TripetData](
    id='chatcmpl-BFxzxDvkhoGvuTVsa5PLVoPh73dMK',
    choices=[
        ParsedChoice[TripetData](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[TripetData](
                content='{"triplets":[{"query":"Naon anu kedah dilakukeun sangkan teu jadi 
hambar?","positive_passage":"Sing jembar sabar tong jadi hambar ihtér raksa rasa ukir 
pikir.","negative_passage":"Dulur néang halal awur amal ulah lieur ku madu dunya."},{"query":"Kumaha cara ngajaga 
pikiran sangkan cageur?","positive_passage":"Mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi jalma
leutik.","negative_passage":"Abong di gedong lain batur kabéh gelé."},{"query":"Naon anu jadi penting pikeun 
ngahontal kahayang?","positive_passage":"Balukar janglar meunang kahayang balukar sasar pedar 
ringkang.","negative_passage":"Ulah sombong, jung sanding ka nu agung gusti."},{"query":"Kumaha cara pikeun pariksa
ati?","positive_passage":"Pariksa ati sangkan ngarti nu sajati.","negative_passage":"Rasa ukir pikir ieu 
mencerminkan perjalanan hirup."},{"query":"Naon pesen pikeun hirup di dunya?","positive_passage":"Ulah lieur ku 
madu dunya, tong jadi jalma leutik.","negative_passage":"Ihtér raksa rasa bisa jadi pituduh lamun 
dilaksanakeun."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=TripetData(
                    triplets=[
                        TripletItem(
                            query='Naon anu kedah dilakukeun sangkan teu jadi hambar?',
                            positive_passage='Sing jembar sabar tong jadi hambar ihtér raksa rasa ukir pikir.',
                            negative_passage='Dulur néang halal awur amal ulah lieur ku madu dunya.'
                        ),
                        TripletItem(
                            query='Kumaha cara ngajaga pikiran sangkan cageur?',
                            positive_passage='Mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi 
jalma leutik.',
                            negative_passage='Abong di gedong lain batur kabéh gelé.'
                        ),
                        TripletItem(
                            query='Naon anu jadi penting pikeun ngahontal kahayang?',
                            positive_passage='Balukar janglar meunang kahayang balukar sasar pedar ringkang.',
                            negative_passage='Ulah sombong, jung sanding ka nu agung gusti.'
                        ),
                        TripletItem(
                            query='Kumaha cara pikeun pariksa ati?',
                            positive_passage='Pariksa ati sangkan ngarti nu sajati.',
                            negative_passage='Rasa ukir pikir ieu mencerminkan perjalanan hirup.'
                        ),
                        TripletItem(
                            query='Naon pesen pikeun hirup di dunya?',
                            positive_passage='Ulah lieur ku madu dunya, tong jadi jalma leutik.',
                            negative_passage='Ihtér raksa rasa bisa jadi pituduh lamun dilaksanakeun.'
                        )
                    ]
                ),
                annotations=[]
            )
        )
    ],
    created=1743145253,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_b376dfbbd5',
    usage=CompletionUsage(
        completion_tokens=274,
        prompt_tokens=363,
        total_tokens=637,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## Generate OpenAI Batch Request

### Batch Request Generator

In [43]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel):
    for row in df.itertuples():
        custom_id = str(uuid4())
        job_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": row.content},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": base_model.__name__,
                        "strict": True,
                        "schema": to_strict_json_schema(base_model),
                    },
                },
            },
        }
        
        yield (custom_id, row.id, job_data)

In [44]:
next(generate_batch(df, SYSTEM_PROMPTS["BEIR"], BEIRQuery))

('e9ffd3ed-da24-4e8b-90c8-7a54bfffd97d',
 '0f438470-de7f-47cb-8ba2-16e8b1ff5750',
 {'custom_id': 'e9ffd3ed-da24-4e8b-90c8-7a54bfffd97d',
  'method': 'POST',
  'url': '/v1/chat/completions',
  'body': {'model': 'gpt-4o-mini',
   'messages': [{'role': 'system',
     'content': 'Generate 5 short search query-answer pairs based on the provided document. Each query should resemble a natural search input (as if searching on Google or other search engines).\n- The query and the answer must be written in Sundanese.\n- Ensure the response is concise, accurate, and directly relevant to the document.'},
    {'role': 'user',
     'content': 'bismillah yuga lampah balukar janglar meunang kahayang balukar sasar pedar ringkang sing jembar sabar tong jadi hambar ihtér raksa rasa ukir pikir mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi jalma leutik ulah sombong abong di gedong lain batur kabéh gelé dulur néang halal awur amal ulah lieur ku madu dunya jung sanding ka nu agung gusti wé

In [45]:
def persist_batch(kind: str, schema: BaseModel):
    batch_req_path = Path(f"../data/{kind}/{kind}_batch.jsonl")
    batch_req_path.parent.mkdir(parents=True, exist_ok=True)

    batch_map_path = Path(f"../data/{kind}/{kind}_map.jsonl")
    batch_map_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(batch_req_path, "w") as fm, open(batch_map_path, "w") as mm:
        batch_iter = generate_batch(df, SYSTEM_PROMPTS[kind.upper()], schema)
        for custom_id, doc_id, req in batch_iter:
            json.dump(req, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_id": doc_id}, mm)
            mm.write("\n")
    
    return batch_req_path, batch_map_path

In [46]:
beir_req_path, beir_map_path = persist_batch("beir", BEIRQuery)
triplet_req_path, triplet_map_path = persist_batch("triplet", TripetData)

### Submit Batch Requests

In [47]:
def submit_batch(path):
    batch_file = client.files.create(file=open(path, "rb"), purpose="batch")

    return client.batches.create(
        input_file_id=batch_file.id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h"
    )

In [50]:
beir_batch = submit_batch(beir_req_path.resolve())
print(beir_batch)

Batch(
    id='batch_67e64f6818d48190b7572af05d71fa35',
    completion_window='24h',
    created_at=1743146856,
    endpoint='/v1/chat/completions',
    input_file_id='file-Uveu2sbk5LjLCfpczKzpve',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1743233256,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [51]:
triplet_batch = submit_batch(triplet_req_path.resolve())
print(triplet_batch)

Batch(
    id='batch_67e64f8db3988190a7a04356370a1af5',
    completion_window='24h',
    created_at=1743146893,
    endpoint='/v1/chat/completions',
    input_file_id='file-KzHK8pZe8AggMc7ARmqRFy',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1743233293,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)